# B4 — Diabetes Prediction: KNN and classification metrics

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/mragpavank/pima-indians-diabetes-database/data

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'diabetes.csv'

## Implementation and evaluation

In [ ]:
"""B4: Diabetes KNN with confusion matrix, accuracy, error, precision and recall."""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score,recall_score

def main():
    if CSV_PATH:  # Load diabetes.csv here (Pima dataset with Outcome column).
        df=pd.read_csv(CSV_PATH)
    else:
        x,y=make_classification(n_samples=500,n_features=8,n_informative=5,random_state=42)
        df=pd.DataFrame(x,columns=[f'feature_{i}' for i in range(8)]);df['Outcome']=y
        print('Using synthetic demonstration data.')
    target=next((c for c in ['Outcome','outcome','diabetes'] if c in df),None)
    if target is None:raise ValueError('Expected Outcome column')
    df=df.dropna(subset=[target]);y=pd.to_numeric(df[target],errors='raise').astype(int)
    X=df.drop(columns=[target]).apply(pd.to_numeric,errors='coerce')
    if y.nunique()!=2 or y.value_counts().min()<2:raise ValueError('Need both classes')
    # Pima medical zeros can represent missing readings, but not all zeros are missing.
    for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
        if col in X:X.loc[X[col]==0,col]=np.nan
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
    model=Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=5))])
    model.fit(Xtr,ytr);pred=model.predict(Xte)
    acc=accuracy_score(yte,pred)
    print('Confusion matrix:\n',confusion_matrix(yte,pred,labels=[0,1]))
    print(f'Accuracy={acc:.4f} Error rate={1-acc:.4f} Precision={precision_score(yte,pred,zero_division=0):.4f} Recall={recall_score(yte,pred,zero_division=0):.4f}')
main()
